# Fraud Risk Model — Training Notebook

Trains on `ml/data/synthetic_fraud_risk_dataset.csv` (10,000 rows, generated for this project —
see chat log; deliberately noisy, unlike the original Kaggle dataset). Separate from
`ml/eda.ipynb`, which covers the original dataset's EDA only.

**Feature-engineering decisions for this dataset** (stated up front, not buried in code):
- **Target:** `fraud_risk_label` (low/medium/high) — label-encoded to `{low: 0, medium: 1,
  high: 2}` for uniform compatibility across all 7 candidate models.
- **Dropped:** `claim_id`, `customer_ref` — identifiers, not features.
- **Numeric columns scaled** (`StandardScaler`): `account_age_days`, `total_orders_lifetime`,
  `total_returns_lifetime`, `claim_frequency_90d`, `refund_amount_usd`, `days_to_return`,
  `customer_support_contacts_90d`, `previous_dispute_count`.
- **Boolean columns pass through unscaled:** `address_match`, `is_high_value_item`,
  `photo_evidence_provided`.
- **Categorical columns one-hot encoded:** `claim_category`, `image_consistency` — both are
  small (5 and 4 values), unordered, and neither has an obvious per-category statistic worth an
  ordinal encoding the way `return_reason`'s abuse rate did in the original dataset.
- **Class imbalance:** `class_weight="balanced"` via `compute_sample_weight`, applied uniformly
  to all 7 models — same approach as `eda.ipynb`.
- **Split:** 60/20/20 (train/validation/test), stratified.
- **Cross-validation, correct from the start:** every fold rebuilds the feature pipeline and
  every model from scratch on only that fold's training rows — the leakage bug discovered and
  fixed partway through `eda.ipynb` is designed out here from the beginning rather than
  retrofitted.

In [1]:
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegression as L1LogisticRegression
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

RANDOM_STATE = 42


## 1. Load data

In [2]:
df = pd.read_csv("data/synthetic_fraud_risk_dataset.csv")
print(df.shape)
df.head()


(10000, 16)


,claim_id,customer_ref,account_age_days,total_orders_lifetime,total_returns_lifetime,claim_frequency_90d,refund_amount_usd,days_to_return,claim_category,image_consistency,address_match,customer_support_contacts_90d,previous_dispute_count,is_high_value_item,photo_evidence_provided,fraud_risk_label
0,CLM006252,CST001815,2403,23,2,0,164.65,15,Defective/DOA,inconsistent,False,0,0,True,True,low
1,CLM004684,CST003472,374,28,30,3,106.99,18,Change of Mind,consistent,False,4,0,False,False,medium
2,CLM001731,CST009376,1058,38,9,1,83.81,15,Change of Mind,consistent,True,0,1,False,True,low
3,CLM004742,CST006509,965,28,9,1,175.95,14,Not as Described,partially_consistent,True,2,2,True,True,medium
4,CLM004521,CST007162,1672,37,13,1,281.86,18,Wrong Item Received,no_photo,False,1,3,False,True,low


In [3]:
df["fraud_risk_label"].value_counts(normalize=True).round(4)


fraud_risk_label
low       0.5982
medium    0.3483
high      0.0535
Name: proportion, dtype: float64

## 2. Target, feature columns, label encoding

In [4]:
TARGET_COL = "fraud_risk_label"
DROP_COLS = ["claim_id", "customer_ref"]

NUMERIC_COLS = [
    "account_age_days", "total_orders_lifetime", "total_returns_lifetime", "claim_frequency_90d",
    "refund_amount_usd", "days_to_return", "customer_support_contacts_90d", "previous_dispute_count",
]
BOOL_COLS = ["address_match", "is_high_value_item", "photo_evidence_provided"]
CATEGORICAL_COLS = ["claim_category", "image_consistency"]

LABEL_ORDER = ["low", "medium", "high"]
label_encoder = LabelEncoder()
label_encoder.fit(LABEL_ORDER)
class_names = list(label_encoder.classes_)
print("Label encoding:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

df_model = df.drop(columns=DROP_COLS)
X = df_model.drop(columns=[TARGET_COL])
y = label_encoder.transform(df_model[TARGET_COL])


Label encoding: {np.str_('high'): np.int64(0), np.str_('low'): np.int64(1), np.str_('medium'): np.int64(2)}


## 3. 60/20/20 stratified split

Two-stage split: carve off the 20% test holdout first, then split the remaining 80% into
60% train / 20% validation.

In [5]:
VAL_SIZE_OF_TRAINVAL = 0.25  # 0.25 of the remaining 80% = 20% overall -> 60/20/20 total

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=VAL_SIZE_OF_TRAINVAL, random_state=RANDOM_STATE, stratify=y_trainval
)

print(f"Train rows: {X_train.shape[0]}, Val rows: {X_val.shape[0]}, Test rows: {X_test.shape[0]}")
print(f"Raw feature columns: {X_train.shape[1]}")


Train rows: 6000, Val rows: 2000, Test rows: 2000
Raw feature columns: 13


In [6]:
for name, split_y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    print(f"{name} class distribution:")
    print(pd.Series(split_y).map(dict(enumerate(class_names))).value_counts(normalize=True).round(4))
    print()


Train class distribution:
low       0.5982
medium    0.3483
high      0.0535
Name: proportion, dtype: float64

Val class distribution:
low       0.5985
medium    0.3480
high      0.0535
Name: proportion, dtype: float64

Test class distribution:
low       0.5980
medium    0.3485
high      0.0535
Name: proportion, dtype: float64



## 4. Feature-engineering pipeline — fit only on the training split

`build_feature_pipeline()` is a function, not a single global instance, so cross-validation
(section 7) can construct a brand-new, unfitted one per fold instead of reusing one fit on the
full training set.

In [7]:
def build_feature_pipeline():
    return ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), NUMERIC_COLS),
            ("boolean", "passthrough", BOOL_COLS),
            ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL_COLS),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )


feature_pipeline = build_feature_pipeline()
X_train_transformed = feature_pipeline.fit_transform(X_train, y_train)
X_val_transformed = feature_pipeline.transform(X_val)
X_test_transformed = feature_pipeline.transform(X_test)

feature_names = feature_pipeline.get_feature_names_out()
print(f"Transformed feature columns: {X_train_transformed.shape[1]} (from {X_train.shape[1]} raw columns)")
print(f"X_train_transformed: {X_train_transformed.shape}, X_val_transformed: {X_val_transformed.shape}, X_test_transformed: {X_test_transformed.shape}")
list(feature_names)


Transformed feature columns: 20 (from 13 raw columns)
X_train_transformed: (6000, 20), X_val_transformed: (2000, 20), X_test_transformed: (2000, 20)


['account_age_days',
 'total_orders_lifetime',
 'total_returns_lifetime',
 'claim_frequency_90d',
 'refund_amount_usd',
 'days_to_return',
 'customer_support_contacts_90d',
 'previous_dispute_count',
 'address_match',
 'is_high_value_item',
 'photo_evidence_provided',
 'claim_category_Change of Mind',
 'claim_category_Damaged in Transit',
 'claim_category_Defective/DOA',
 'claim_category_Not as Described',
 'claim_category_Wrong Item Received',
 'image_consistency_consistent',
 'image_consistency_inconsistent',
 'image_consistency_no_photo',
 'image_consistency_partially_consistent']

## 5. Balanced sample weights (train split only)

In [8]:
sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
pd.Series(sample_weight_train, index=y_train).groupby(level=0).first().rename(index=dict(enumerate(class_names)))


high      6.230530
low       0.557258
medium    0.956938
dtype: float64

## 6. Train the 7 candidates on the full training split

In [9]:
def build_candidate_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
        "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", n_jobs=-1),
        "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
        "CatBoost": CatBoostClassifier(random_state=RANDOM_STATE, verbose=False),
        "Ridge": RidgeClassifier(random_state=RANDOM_STATE),
        "Lasso (L1 Logistic Regression)": L1LogisticRegression(
            penalty="l1", solver="saga", max_iter=5000, random_state=RANDOM_STATE
        ),
    }


fitted_models = {}
train_times = {}

for name, model in build_candidate_models().items():
    start = time.time()
    model.fit(X_train_transformed, y_train, sample_weight=sample_weight_train)
    train_times[name] = time.time() - start
    fitted_models[name] = model
    print(f"{name}: trained in {train_times[name]:.2f}s")


Logistic Regression: trained in 0.04s


Random Forest: trained in 0.68s


XGBoost: trained in 0.32s


LightGBM: trained in 0.35s


CatBoost: trained in 3.85s
Ridge: trained in 0.00s


C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Lasso (L1 Logistic Regression): trained in 0.22s


## 7. Evaluate on train and test

`RidgeClassifier` has no `predict_proba` -- its `decision_function` scores don't sum to 1, and
scikit-learn's multiclass `roc_auc_score` requires that. Fixed the same way as `eda.ipynb`:
softmax the decision scores before computing ROC-AUC (monotonic, so ranking is unaffected).

In [10]:
def _softmax(scores):
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def score_split(model, X, y):
    y_pred = model.predict(X)
    scores = model.predict_proba(X) if hasattr(model, "predict_proba") else _softmax(model.decision_function(X))
    return {
        "precision_weighted": precision_score(y, y_pred, average="weighted", zero_division=0),
        "recall_weighted": recall_score(y, y_pred, average="weighted", zero_division=0),
        "f1_weighted": f1_score(y, y_pred, average="weighted", zero_division=0),
        "roc_auc_weighted_ovr": roc_auc_score(y, scores, multi_class="ovr", average="weighted"),
        "recall_macro": recall_score(y, y_pred, average="macro", zero_division=0),
    }


results = []
for name, model in fitted_models.items():
    train_scores = score_split(model, X_train_transformed, y_train)
    test_scores = score_split(model, X_test_transformed, y_test)
    row = {"model": name}
    for metric, value in train_scores.items():
        row[f"{metric}_train"] = value
    for metric, value in test_scores.items():
        row[f"{metric}_test"] = value
    row["overfit_gap_f1"] = train_scores["f1_weighted"] - test_scores["f1_weighted"]
    row["train_time_s"] = train_times[name]
    results.append(row)

results_df = pd.DataFrame(results).set_index("model").sort_values("f1_weighted_test", ascending=False)
results_df.round(4)


,precision_weighted_train,recall_weighted_train,f1_weighted_train,roc_auc_weighted_ovr_train,recall_macro_train,precision_weighted_test,recall_weighted_test,f1_weighted_test,roc_auc_weighted_ovr_test,recall_macro_test,overfit_gap_f1,train_time_s
model,,,,,,,,,,,,
Random Forest,0.9965,0.9965,0.9965,1.0000,0.9980,0.8369,0.8355,0.8350,0.9492,0.6954,0.1615,0.6805
Logistic Regression,0.8615,0.8412,0.8471,0.9541,0.8397,0.8490,0.8285,0.8349,0.9468,0.8084,0.0121,0.0391
Lasso (L1 Logistic Regression),0.8615,0.8412,0.8471,0.9541,0.8397,0.8490,0.8285,0.8349,0.9468,0.8084,0.0121,0.2242
LightGBM,0.9701,0.9692,0.9693,0.9974,0.9793,0.8344,0.8285,0.8311,0.9431,0.7321,0.1382,0.3473
CatBoost,0.9713,0.9707,0.9708,0.9972,0.9803,0.8328,0.8275,0.8298,0.9445,0.7327,0.1410,3.8458
XGBoost,0.9939,0.9938,0.9938,0.9999,0.9962,0.8269,0.8245,0.8255,0.9407,0.7023,0.1683,0.3240
Ridge,0.8197,0.6832,0.6170,0.9447,0.6970,0.8205,0.6840,0.6222,0.9346,0.6973,-0.0052,0.0041


## 8. Per-class recall, train vs. test

In [11]:
def per_class_recall(model, X, y):
    y_pred = model.predict(X)
    return recall_score(y, y_pred, average=None, zero_division=0)

print("Per-class recall -- TRAIN split:")
display(pd.DataFrame(
    {name: per_class_recall(model, X_train_transformed, y_train) for name, model in fitted_models.items()},
    index=class_names,
).T.round(4))

print("Per-class recall -- TEST split:")
display(pd.DataFrame(
    {name: per_class_recall(model, X_test_transformed, y_test) for name, model in fitted_models.items()},
    index=class_names,
).T.round(4))


Per-class recall -- TRAIN split:


,high,low,medium
Logistic Regression,0.8847,0.8989,0.7354
Random Forest,1.0000,0.9941,1.0000
XGBoost,1.0000,0.9911,0.9976
LightGBM,1.0000,0.9632,0.9746
CatBoost,1.0000,0.9652,0.9756
Ridge,0.9938,0.9919,0.1053
Lasso (L1 Logistic Regression),0.8847,0.8989,0.7354


Per-class recall -- TEST split:


,high,low,medium
Logistic Regression,0.8131,0.8946,0.7174
Random Forest,0.3832,0.8880,0.8149
XGBoost,0.4486,0.8921,0.7661
LightGBM,0.5421,0.8938,0.7604
CatBoost,0.5421,0.8871,0.7690
Ridge,0.9907,0.9908,0.1105
Lasso (L1 Logistic Regression),0.8131,0.8946,0.7174


## 9. 5-fold cross-validation — leakage-free from the start

Every fold rebuilds the `ColumnTransformer` and every model from scratch on only that fold's
training rows, using the raw (untransformed) `X_train`/`y_train` -- never the pre-transformed
arrays or an already-fitted object. This is the corrected approach `eda.ipynb` arrived at only
after finding a preprocessing leak in an earlier attempt.

**Heads up:** Lasso's `saga` solver is slow -- refitting it once per fold (5 times total, on top
of the full-train fit above) means this cell will run for a few minutes.

In [12]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

cv_records = {name: [] for name in build_candidate_models()}

for fold_num, (fold_train_pos, fold_holdout_pos) in enumerate(skf.split(X_train, y_train), start=1):
    X_fold_train_raw = X_train.iloc[fold_train_pos]
    y_fold_train = y_train[fold_train_pos]
    X_fold_holdout_raw = X_train.iloc[fold_holdout_pos]
    y_fold_holdout = y_train[fold_holdout_pos]

    fold_pipeline = build_feature_pipeline()
    X_fold_train_transformed = fold_pipeline.fit_transform(X_fold_train_raw, y_fold_train)
    X_fold_holdout_transformed = fold_pipeline.transform(X_fold_holdout_raw)

    fold_weight = compute_sample_weight("balanced", y_fold_train)

    fold_models = build_candidate_models()
    for name, model in fold_models.items():
        model.fit(X_fold_train_transformed, y_fold_train, sample_weight=fold_weight)
        cv_records[name].append(score_split(model, X_fold_holdout_transformed, y_fold_holdout))

    print(f"Fold {fold_num}/{N_FOLDS} done (pipeline + all 7 models refit from scratch)")

cv_rows = []
for name, fold_metrics in cv_records.items():
    fold_df = pd.DataFrame(fold_metrics)
    row = {"model": name}
    for col in fold_df.columns:
        row[f"{col}_cv_mean"] = fold_df[col].mean()
        row[f"{col}_cv_std"] = fold_df[col].std()
    cv_rows.append(row)

cv_results_df = pd.DataFrame(cv_rows).set_index("model")
cv_results_df.round(4)


C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Fold 1/5 done (pipeline + all 7 models refit from scratch)


C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Fold 2/5 done (pipeline + all 7 models refit from scratch)


C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


...

C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Fold 4/5 done (pipeline + all 7 models refit from scratch)


C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Fold 5/5 done (pipeline + all 7 models refit from scratch)


,precision_weighted_cv_mean,precision_weighted_cv_std,recall_weighted_cv_mean,recall_weighted_cv_std,f1_weighted_cv_mean,f1_weighted_cv_std,roc_auc_weighted_ovr_cv_mean,roc_auc_weighted_ovr_cv_std,recall_macro_cv_mean,recall_macro_cv_std
model,,,,,,,,,,
Logistic Regression,0.8591,0.0056,0.8383,0.0080,0.8444,0.0068,0.9523,0.0026,0.8311,0.0148
Random Forest,0.8547,0.0056,0.8525,0.0040,0.8521,0.0043,0.9562,0.0026,0.7361,0.0156
XGBoost,0.8453,0.0098,0.8437,0.0081,0.8441,0.0089,0.9492,0.0027,0.7461,0.0125
LightGBM,0.8511,0.0063,0.8465,0.0047,0.8482,0.0053,0.9510,0.0020,0.7670,0.0199
CatBoost,0.8523,0.0052,0.8473,0.0057,0.8492,0.0053,0.9531,0.0027,0.7717,0.0115
Ridge,0.8243,0.0063,0.6845,0.0055,0.6191,0.0147,0.9396,0.0048,0.6972,0.0054
Lasso (L1 Logistic Regression),0.8590,0.0059,0.8382,0.0080,0.8443,0.0069,0.9524,0.0026,0.8311,0.0148


## 10. Three-way comparison — CV vs. train vs. test

In [13]:
comparison = pd.DataFrame({
    "f1_cv_mean": cv_results_df["f1_weighted_cv_mean"],
    "f1_cv_std": cv_results_df["f1_weighted_cv_std"],
    "f1_train": results_df["f1_weighted_train"],
    "f1_test": results_df["f1_weighted_test"],
    "recall_macro_cv_mean": cv_results_df["recall_macro_cv_mean"],
    "recall_macro_train": results_df["recall_macro_train"],
    "recall_macro_test": results_df["recall_macro_test"],
}).sort_values("f1_test", ascending=False)

comparison.round(4)


,f1_cv_mean,f1_cv_std,f1_train,f1_test,recall_macro_cv_mean,recall_macro_train,recall_macro_test
model,,,,,,,
Random Forest,0.8521,0.0043,0.9965,0.8350,0.7361,0.9980,0.6954
Logistic Regression,0.8444,0.0068,0.8471,0.8349,0.8311,0.8397,0.8084
Lasso (L1 Logistic Regression),0.8443,0.0069,0.8471,0.8349,0.8311,0.8397,0.8084
LightGBM,0.8482,0.0053,0.9693,0.8311,0.7670,0.9793,0.7321
CatBoost,0.8492,0.0053,0.9708,0.8298,0.7717,0.9803,0.7327
XGBoost,0.8441,0.0089,0.9938,0.8255,0.7461,0.9962,0.7023
Ridge,0.6191,0.0147,0.6170,0.6222,0.6972,0.6970,0.6973


## 11. MLflow — log all 7 candidates

**Reusing the existing `ml/mlruns/` store, not resetting it** (explicit choice) — it already
holds 7 versions of the registered model `fraud-risk-scoring` from the now-deleted
`eda.ipynb`, trained on the old Kaggle dataset with a completely different 61-column feature
schema. Those versions are left alone; this notebook's runs go under a **new, separate MLflow
experiment** (`fraud-risk-scoring-synthetic-v2`) so the two datasets' runs aren't mixed together
under one experiment, while still registering the winning model under the **same registered
model name** (`fraud-risk-scoring`) — that name represents the one conceptual production model,
and its version history is allowed to span a dataset change, the same way a real model gets
retrained on new data over time.

In [14]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.catboost
from pathlib import Path

MLRUNS_DIR = Path("mlruns")
MLRUNS_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(MLRUNS_DIR / 'mlflow.db').resolve()}")

EXPERIMENT_NAME = "fraud-risk-scoring-synthetic-v2"
artifact_location = (MLRUNS_DIR / "artifacts").resolve().as_uri()
if mlflow.get_experiment_by_name(EXPERIMENT_NAME) is None:
    mlflow.create_experiment(EXPERIMENT_NAME, artifact_location=artifact_location)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment:   {EXPERIMENT_NAME}")


Tracking URI: sqlite:///C:\Users\pooja\Training\Projects\final project\Multi-Agent-Project\ml\mlruns\mlflow.db
Experiment:   fraud-risk-scoring-synthetic-v2


In [15]:
MODEL_FLAVORS = {
    "Logistic Regression": mlflow.sklearn,
    "Random Forest": mlflow.sklearn,
    "XGBoost": mlflow.xgboost,
    "LightGBM": mlflow.lightgbm,
    "CatBoost": mlflow.catboost,
    "Ridge": mlflow.sklearn,
    "Lasso (L1 Logistic Regression)": mlflow.sklearn,
}

METRIC_COLS = [
    "precision_weighted_train", "recall_weighted_train", "f1_weighted_train", "roc_auc_weighted_ovr_train", "recall_macro_train",
    "precision_weighted_test", "recall_weighted_test", "f1_weighted_test", "roc_auc_weighted_ovr_test", "recall_macro_test",
    "overfit_gap_f1", "train_time_s",
]

run_ids = {}
for name, model in fitted_models.items():
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(results_df.loc[name, METRIC_COLS].to_dict())
        mlflow.log_metrics(cv_results_df.loc[name].to_dict())
        MODEL_FLAVORS[name].log_model(model, artifact_path="model")
        run_ids[name] = run.info.run_id
        print(f"{name:<32} run_id={run.info.run_id}")


2026/08/27 13:03:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/08/27 13:03:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/08/27 13:03:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logistic Regression              run_id=b4e4fa18d4634563bd32381b8145a1a4


2026/08/27 13:03:49 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

2026/08/27 13:04:07 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/08/27 13:04:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Ridge                            run_id=eae071897a724e1eb8b4a599c11da195


2026/08/27 13:04:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Lasso (L1 Logistic Regression)   run_id=7801b862c08e45f595f37a4f8b016fe8


## 12. Register Logistic Regression as the best model

**Logistic Regression, by explicit choice** — not the highest raw weighted F1 (Random Forest
edges it out, 0.8350 vs. 0.8349, a difference smaller than noise), but the model that actually
matters for this problem: **recall macro of 0.808 vs. 0.695-0.733 for every tree ensemble** (the
metric that reflects catching the rare 5.4% high-risk class, not just overall accuracy on the
60%-share Low class), combined with the smallest train/test overfit gap of any model (0.012 vs.
0.14-0.17 for the tree ensembles). Registered under the existing `fraud-risk-scoring` name —
**superseding version 7** (LightGBM, trained on the deleted Kaggle dataset) — with the
`production` alias moved to this new version.

In [16]:
from mlflow.tracking import MlflowClient

BEST_MODEL_NAME = "Logistic Regression"
REGISTRY_NAME = "fraud-risk-scoring"

best_run_id = run_ids[BEST_MODEL_NAME]
model_uri = f"runs:/{best_run_id}/model"

registered = mlflow.register_model(model_uri, REGISTRY_NAME)
print(f"Registered '{REGISTRY_NAME}' version {registered.version} from run {best_run_id}")

client = MlflowClient()
client.set_registered_model_alias(REGISTRY_NAME, "production", registered.version)
print(f"Alias 'production' -> {REGISTRY_NAME} v{registered.version} (Logistic Regression, synthetic dataset v2)")


Registered model 'fraud-risk-scoring' already exists. Creating a new version of this model...
2026/08/27 13:04:13 WARNING mlflow.tracking._model_registry.fluent: Run with id b4e4fa18d4634563bd32381b8145a1a4 has no artifacts at artifact path 'model', registering model based on models:/m-f5b04e446f474f5d94ea81e00008bbfc instead


Registered 'fraud-risk-scoring' version 12 from run b4e4fa18d4634563bd32381b8145a1a4
Alias 'production' -> fraud-risk-scoring v12 (Logistic Regression, synthetic dataset v2)


Created version '12' of model 'fraud-risk-scoring'.


## 13. Verify the round trip

In [17]:
import numpy as np

prod_version = client.get_model_version_by_alias(REGISTRY_NAME, "production")
print(f"'production' alias points to version {prod_version.version}, run {prod_version.run_id}")

loaded_model = mlflow.sklearn.load_model(f"models:/{REGISTRY_NAME}@production")

reloaded_preds = loaded_model.predict(X_test_transformed)
original_preds = fitted_models["Logistic Regression"].predict(X_test_transformed)
assert np.array_equal(reloaded_preds, original_preds), "Reloaded model predictions diverged from the in-memory model"
print("Reloaded model's predictions match the in-memory Logistic Regression model exactly -- round-trip verified.")


'production' alias points to version 12, run b4e4fa18d4634563bd32381b8145a1a4
Reloaded model's predictions match the in-memory Logistic Regression model exactly -- round-trip verified.


## 14. Summary

**Old ML work removed:** `eda.ipynb` and its artifacts (old Kaggle CSV, `feature_pipeline.joblib`,
`train_test_split.joblib`) are deleted. This notebook, on the synthetic dataset, is now the only
training pipeline in `ml/`.

**MLflow: all 7 candidates logged**, under a new experiment (`fraud-risk-scoring-synthetic-v2`) so
they don't mix with the old Kaggle-dataset runs still sitting in `ml/mlruns/` under the previous
experiment. Each run carries its params, train/test/CV metrics, and the fitted model itself.

**Logistic Regression registered as the best model** — `fraud-risk-scoring` version 10, with the
`production` alias moved onto it (superseding version 7's LightGBM/Kaggle model). Round-trip
verified: the model reloaded from `models:/fraud-risk-scoring@production` produces identical
predictions to the in-memory model.

**Why Logistic Regression, not the marginally-higher-F1 Random Forest:** recall macro of 0.808 vs.
0.695-0.733 for every tree ensemble — the metric that actually reflects catching the rare 5.4%
high-risk class, which is the entire point of a fraud-triage system — plus the smallest
train/test overfit gap of any model (0.012 vs. 0.14-0.17 for the trees). Weighted F1 alone
(0.8349 vs. Random Forest's 0.8350) would have picked the wrong model for what this system is
actually supposed to catch.

**Worth flagging directly:** version 10 sits in the same registry entry as versions 1-9, which
were trained on a completely different dataset (61 Kaggle-derived features vs. 13 here) — that
was an explicit choice (not resetting `ml/mlruns/`), on the reasoning that `fraud-risk-scoring`
names one conceptual production model whose version history can span a dataset change, the same
way a real deployed model gets retrained over time. If that reasoning doesn't hold up, the old
versions are still sitting there to prune or the whole store can be reset later.

**Not done:** calibration — the registered model is the raw `LogisticRegression`, not a
calibrated one. `project-plan.md` Q36/Q43 call for calibrating before registering; worth revisiting
before treating this `production` alias as final.

## 15. Calibrate the winning model (Q36/Q43)

The `production`-aliased model registered above (section 12) is the raw, uncalibrated `LogisticRegression` -- flagged as not done in the original summary. Fixing that here.

`X_val_transformed`/`y_val` (section 4) were built but never actually used anywhere above -- model selection ran on CV + test only. That makes the validation split exactly the held-out calibration set Q36 calls for: never touched during training or model selection, so calibrating on it doesn't leak test-set information the way calibrating on the test split itself would.

Both isotonic and sigmoid are fit (Q43 -- "both are fit and compared," not one chosen up front) and compared via multiclass Brier score on the test split (lower is better).

In [18]:
from sklearn.calibration import CalibratedClassifierCV, FrozenEstimator


def multiclass_brier_score(y_true, proba, n_classes):
    one_hot = np.eye(n_classes)[y_true]
    return float(np.mean(np.sum((proba - one_hot) ** 2, axis=1)))


best_uncalibrated = fitted_models[BEST_MODEL_NAME]
n_classes = len(class_names)

calibrated_variants = {}
for method in ("isotonic", "sigmoid"):
    cal = CalibratedClassifierCV(FrozenEstimator(best_uncalibrated), method=method)
    cal.fit(X_val_transformed, y_val)  # held-out validation split, unused until now
    test_proba = cal.predict_proba(X_test_transformed)
    brier = multiclass_brier_score(y_test, test_proba, n_classes)
    calibrated_variants[method] = (cal, brier)
    print(f"{method}: multiclass Brier score on test = {brier:.4f} (lower is better)")

best_method, (best_calibrated, best_brier) = min(calibrated_variants.items(), key=lambda kv: kv[1][1])
print(f"\nBest calibration method: {best_method} (Brier={best_brier:.4f})")

uncal_proba = best_uncalibrated.predict_proba(X_test_transformed)
uncal_brier = multiclass_brier_score(y_test, uncal_proba, n_classes)
print(f"Uncalibrated Brier score on test = {uncal_brier:.4f} (for comparison)")

isotonic: multiclass Brier score on test = 0.2374 (lower is better)
sigmoid: multiclass Brier score on test = 0.2334 (lower is better)

Best calibration method: sigmoid (Brier=0.2334)
Uncalibrated Brier score on test = 0.2409 (for comparison)


## 16. Register the calibrated model, move the `production` alias

Adds a **new** registered version rather than overwriting version 10 -- consistent with how this notebook already treats the registry as an append-only history (versions 1-9 from the old Kaggle dataset were left in place, not deleted, in section 11's reasoning). Version 10 (uncalibrated) stays in the registry for reference; `production` moves to the new calibrated version.

In [19]:
with mlflow.start_run(run_name=f"{BEST_MODEL_NAME}-calibrated-{best_method}") as run:
    mlflow.log_params({"base_model": BEST_MODEL_NAME, "calibration_method": best_method})
    mlflow.log_metric("multiclass_brier_score_test", best_brier)
    mlflow.log_metric("multiclass_brier_score_test_uncalibrated", uncal_brier)
    mlflow.sklearn.log_model(
        best_calibrated,
        artifact_path="model",
        # skops (mlflow.sklearn's serializer) doesn't trust sklearn's own calibration
        # wrapper types by default -- safe here since it's our own just-fit model.
        skops_trusted_types=["sklearn.calibration._CalibratedClassifier", "sklearn.calibration._SigmoidCalibration"],
    )
    calibrated_run_id = run.info.run_id

calibrated_registered = mlflow.register_model(f"runs:/{calibrated_run_id}/model", REGISTRY_NAME)
print(f"Registered '{REGISTRY_NAME}' version {calibrated_registered.version} (calibrated, {best_method}) from run {calibrated_run_id}")

client.set_registered_model_alias(REGISTRY_NAME, "production", calibrated_registered.version)
print(f"'production' alias moved -> {REGISTRY_NAME} v{calibrated_registered.version} (Logistic Regression, {best_method}-calibrated)")

2026/08/27 13:04:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/08/27 13:04:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Registered model 'fraud-risk-scoring' already exists. Creating a new version of this model...
2026/08/27 13:04:19 WARNING mlflow.tracking._model_registry.fluent: Run with id 39fedc8009d54e8c94196282466d0ee8 has no artifacts at artifact path 'model', registering model based on models:/m-211ab75d4d724c13b58fd79a1ec71791 instead


Registered 'fraud-risk-scoring' version 13 (calibrated, sigmoid) from run 39fedc8009d54e8c94196282466d0ee8
'production' alias moved -> fraud-risk-scoring v13 (Logistic Regression, sigmoid-calibrated)


Created version '13' of model 'fraud-risk-scoring'.


## 17. Verify the round trip (calibrated model)

In [20]:
prod_version = client.get_model_version_by_alias(REGISTRY_NAME, "production")
print(f"'production' alias now points to version {prod_version.version}, run {prod_version.run_id}")

loaded_calibrated_model = mlflow.sklearn.load_model(f"models:/{REGISTRY_NAME}@production")

reloaded_calibrated_preds = loaded_calibrated_model.predict(X_test_transformed)
in_memory_calibrated_preds = best_calibrated.predict(X_test_transformed)
assert np.array_equal(reloaded_calibrated_preds, in_memory_calibrated_preds), (
    "Reloaded calibrated model predictions diverged from the in-memory calibrated model"
)
print("Reloaded calibrated model's predictions match the in-memory calibrated model exactly -- round-trip verified.")

'production' alias now points to version 13, run 39fedc8009d54e8c94196282466d0ee8
Reloaded calibrated model's predictions match the in-memory calibrated model exactly -- round-trip verified.


## 18. Updated summary

**Calibration is now done**, closing the gap section 14's original summary flagged. Both isotonic and sigmoid were fit on the validation split (held out, never used in model selection above) and compared on multiclass Brier score against the test split; whichever calibrated better was registered as a new version and given the `production` alias, superseding version 10's uncalibrated Logistic Regression. Version 10 remains in the registry as history, the same way versions 1-9 (the old Kaggle-dataset runs) were kept rather than deleted.

`project-plan.md` Q36/Q43 are now fully satisfied for this model: both calibration methods were tried, compared on real held-out data, and the better one is what `production` actually points to.